In [195]:
import json
import re
import random

import spacy

In [196]:
with open("database_sentenze.json", 'r', encoding = 'utf-8') as j:
    database = json.load(j)

In [197]:
random.seed(37)
sample = random.sample(database, k=10)

for documento in sample:
    print(documento['ID'])
    print(documento['metadati']['tipo_pronuncia'])

16744/2022
Ord.
08951/2022
Ord.
14496/2022
Ord.
35079/2022
Ord.
16418/2022
Ord.
30792/2022
Ord.
35711/2022
Ord.
25077/2022
Ord.
11100/2022
Ord.
20749/2022
Ord.


In [180]:
titoli_globali = set()

In [181]:
nuovo_json = {'ID': 'ID',
              'metadati':'metadati',
              'testo' : 'testo',
              'sezioni' : {}}

In [182]:
testi = []

In [183]:
def estrazione_testo(sentenza):
    testo = sentenza['testo_completo']
    testi.append(testo)

In [184]:
nlp = spacy.load('it_core_news_sm')

In [185]:
def controllo_titoli_centrati(testo):
    righe_centrate = re.findall(r"___.+?___", testo)
    righe_centrate_bene = [riga[3:-3] for riga in righe_centrate]
    feat = "VerbForm"
    for riga in righe_centrate_bene:
        doc =nlp(riga)
        doc_pos = [t.pos_ for t in doc]
        controllo_punti = zip(doc_pos, doc_pos[1:])
        controllo_char_strani = ['X', 'SYM']
        t_doc = [t for t in doc if t.morph.get(feat) == ['Part']]
        if len(doc) > 5:
            continue
        elif riga == 'P.Q.M.' or riga == 'PQM' or riga == 'PQM.':
            titoli_globali.add(riga)
        elif any(x == y == 'PUNCT' for x,y in controllo_punti) or any(x in doc_pos for x in controllo_char_strani):
            continue
        elif len([n_chunk for n_chunk in doc.noun_chunks]) == 1:
            titoli_globali.add(riga)
        elif 1 <= len(t_doc) <= 2 and 'AUX' not in doc_pos:
            titoli_globali.add(riga)
    
    
        


In [186]:
nuovo_database = []

In [ ]:
def segmentazione(testo):

    nuovo_json = {'ID': 'ID',
              'metadati':'metadati',
              'testo' : 'testo',
              'sezioni' : {}}
    
    titoli_documento = []
    for titolo in titoli_globali:
        print(titolo)
        generatore = re.finditer(rf" \. {titolo} [:;\.]|___{titolo}___", testo)
        for match in generatore:
                titoli_documento.append((match.start(), titolo))
    titoli_documento.sort()
    for ind, titolo in enumerate(titoli_documento):
        if ind + 1 == len(titoli_documento):
            nuovo_json['sezioni'][f'sezione_{ind}_{titolo[1]}'] = testo[titolo[0]:]
        else:
            nuovo_json['sezioni'][f'sezione_{ind}_{titolo[1]}'] = testo[titolo[0]:titoli_documento[ind + 1][0]]

    nuovo_database.append(nuovo_json)
    


In [193]:
for documento in sample:
    print(documento['ID'])
    estrazione_testo(documento)

for documento in testi:
    controllo_titoli_centrati(documento)

print(titoli_globali)

for documento in testi:
    segmentazione(documento)


print(nuovo_database)


16744/2022
08951/2022
14496/2022
35079/2022
16418/2022
30792/2022
35711/2022
25077/2022
11100/2022
20749/2022
{'RILEVATO CHE', 'Il Presidente', 'ordinanza', '- controricorrente -', 'P.Q.M.'}
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
ordinanza
- controricorrente -
P.Q.M.
RILEVATO CHE
Il Presidente
or

In [ ]:
controllo_titoli("___maNgo______man4ato______'per questo minerale'______P.Q.M.______PQM______ritenuto opportuno___\n___'per questi motivi'___io sono bello___visto e considerato:______-contro-______ricorrente___è fatta per oggi___Ho mangiato tutto___")

['ADV']
['NOUN']
['PUNCT', 'ADP', 'DET', 'NOUN', 'PUNCT']
['SYM']
['ADV']
['VERB', 'ADJ']
['PUNCT', 'ADP', 'DET', 'NOUN', 'PUNCT']
['VERB', 'CCONJ', 'VERB', 'PUNCT']
['PUNCT']
['ADJ']
['AUX', 'VERB', 'PRON']
['man4ato', "'per questo minerale'", 'P.Q.M.', 'PQM', 'ritenuto opportuno', "'per questi motivi'", 'visto e considerato:']


In [ ]:
for sentenza in database:
    se